# ONNX Basics 

---

In this notebook, we will explore **ONNX (Open Neural Network Exchange)**, an open format for representing machine learning models that enables cross-platform and cross-language deployment.

We will cover:
- What ONNX is and why it exists
- When you would (and wouldn't) need it as a freelance data scientis
- Converting a scikit-learn Pipeline to ONNX format
- Running inference with the ONNX Runtime
- Comparing predictions between the orifinal model and the ONNX model

---

## 1. What is ONNX?
**ONNX (Open Neural Network Exchange)** is an open-source format designed to represent machine learning models in a **framework-agnostic** way.

The core problem it solves:


| Scenario | Pickle / Joblib | ONNX |
| :--- | :--- | :--- |
| Client's production stack is Python | ✅ Works perfectly | ✅ Works |
| Client's production stack is Java / C# / C++ | ❌ Can't load `.joblib` | ✅ Cross-language support |
| Model needs to run on edge devices / mobile | ❌ Needs full Python | ✅ Lightweight runtime |
| Inference speed is critical | Standard speed | ✅ Optimized runtime |

As a freelance data scientist, you will use joblib 90% of the time. But when a client says, "We need this model in our Java backend" or "We need this to run on a device without Python", ONNX is the answer.

### 1.1. How It Works?

> Train in Python (scikit-learn / PyTorch / etc.)   
>   → Convert to ONNX (.onnx file)   
>       → Run anywhere (Python, Java, C#, C++, JavaScript, mobile)

The conversion step translates your model into a **computational graph**, a language-independet description of the mathematical operations your model performs. The **ONNX Runtime** (a high-performance inference engine by Microsoft) can then execute this graph on any platform.

### 1.2. Prerequisites

ONNX requires the following packages:

- `skl2onnx`: Converts scikit-learn models/pipelines to ONNX format
- `onnxruntime`: Microsoft's high-performance engine for running ONNX models

In [1]:
from pathlib import Path

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

In [2]:
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)


---

## 2. Train a Pipeline
We'll reuse the same Iris Pipeline from the previous notebook so we can directly compare the joblib and ONNX approaches.

In [3]:
# Load and split
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build and train the pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])
pipeline.fit(X_train, y_train)

# Baseline predictions (these are our "ground truth" to compare against)
y_pred_sklearn = pipeline.predict(X_test)
sklearn_accuracy = accuracy_score(y_test, y_pred_sklearn)
print(f"scikit-learn Pipeline accuracy: {sklearn_accuracy:.4f}")

scikit-learn Pipeline accuracy: 1.0000



---

## 3. Convert to ONNX
The `skl2onnx` library's `convert_sklearn()` function takes a trained scikit-learn model (or Pipeline) and converts it to ONNX format.

It requires an **initial type**, a description of the input shape and data type that the model expects. This is how ONNX knows the structure of the computational graph.

In [4]:
# Define the input type: a float tensor with 4 features (Iris has 4 features)
# None means the batch size is dynamic (can predict 1 samples or 1000)
initial_type = [("float_input", FloatTensorType([None, X_train.shape[1]]))]

# Convert the pipeline to ONNX
onnx_model = convert_sklearn(pipeline, initial_types=initial_type)

# Save the ONNX model to disk
onnx_path = MODELS_DIR / "iris_pipeline.onnx"
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"ONNX model saved to: {onnx_path}")
print(f"File size:           {onnx_path.stat().st_size / 1024:.1f} KB")

ONNX model saved to: models/iris_pipeline.onnx
File size:           78.5 KB



---

## 4. Run Inference with ONNX Runtime

Now we'll load the `.onnx` file using **ONNX Runtime** and make predictions. Notice that this step doesn't require scikit-learn at all because the ONNX Runtime is a standalon inference engine.

In a real-cross platform scenario, this same `.onnx` file could be loaded in Java, C#, or JavaScript.

In [5]:
# Create an inference session from the saved ONNX file
session = rt.InferenceSession(str(onnx_path))

# Get the input name (we defined it as "float_input" during conversion)
input_name = session.get_inputs()[0].name
print(f"Input name:  {input_name}")
print(f"Input shape: {session.get_inputs()[0].shape}")
print(f"Input type:  {session.get_inputs()[0].type}")

# Get the output names
output_names = [output.name for output in session.get_outputs()]
print(f"Output names: {output_names}")

Input name:  float_input
Input shape: [None, 4]
Input type:  tensor(float)
Output names: ['output_label', 'output_probability']


In [6]:
# Run inference
# ONNX Runtime expects float32 input (not float64, which is NumPy's default)
X_test_float32 = X_test.astype(np.float32)

onnx_predictions = session.run(None, {input_name: X_test_float32})

# onnx_predictions is a list: [predicted_labels, predicted_probabilities]
y_pred_onnx = onnx_predictions[0]  # predicted labels

print(f"ONNX predictions (first 10): {y_pred_onnx[:10]}")
print(f"sklearn predictions (first 10): {y_pred_sklearn[:10]}")

ONNX predictions (first 10): [1 0 2 1 1 0 1 2 1 1]
sklearn predictions (first 10): [1 0 2 1 1 0 1 2 1 1]



---

## 5. Verify

The whole point of ONNX is that it should produce the **same results** as the original model. Let's verify.

In [7]:
# Compare predictions
predictions_match = np.array_equal(y_pred_onnx, y_pred_sklearn)
onnx_accuracy = accuracy_score(y_test, y_pred_onnx)

print(f"scikit-learn accuracy: {sklearn_accuracy:.4f}")
print(f"ONNX Runtime accuracy: {onnx_accuracy:.4f}")
print(f"Predictions identical:  {predictions_match}")

if predictions_match:
    print("\n✅ ONNX model produces identical results to the original Pipeline.")
else:
    # Minor floating-point differences are possible but rare for classification
    n_different = np.sum(y_pred_sklearn != y_pred_onnx)
    print(f"\n⚠️  {n_different} prediction(s) differ — likely due to floating-point precision.")

scikit-learn accuracy: 1.0000
ONNX Runtime accuracy: 1.0000
Predictions identical:  True

✅ ONNX model produces identical results to the original Pipeline.



---

## 6. File Size Comparison

Let's see how the ONNX file compares to the joblib file from the previous notebook.

In [8]:
import joblib as jl

# Save the same pipeline with joblib for comparison
joblib_path = MODELS_DIR / "iris_pipeline.joblib"
jl.dump(pipeline, joblib_path)

print("File Size Comparison")
print("=" * 40)
print(f"{'Format':<25} {'Size (KB)':>10}")
print("-" * 40)
print(f"{'Joblib (.joblib)':<25} {joblib_path.stat().st_size / 1024:>10.1f}")
print(f"{'ONNX (.onnx)':<25} {onnx_path.stat().st_size / 1024:>10.1f}")

File Size Comparison
Format                     Size (KB)
----------------------------------------
Joblib (.joblib)               183.1
ONNX (.onnx)                    78.5



---

## 7. When to Use ONNX (Decision Guide)

| Question | If Yes → | If No → |
| :--- | :--- | :--- |
| Is the production environment Python? | Use **joblib** | Consider **ONNX** |
| Does the client need the model in Java/C#/C++? | Use **ONNX** | Use **joblib** |
| Is inference speed critical (milliseconds matter)? | Consider **ONNX Runtime** | Use **joblib** |
| Is the model a standard scikit-learn estimator? | Both work | Check ONNX compatibility |
| Are you deploying to edge/mobile devices? | Use **ONNX** | Use **joblib** |

**For most freelance data science work:** Start with joblib. Switch to ONNX only when a specific client requirement demands it.

### 7.1. ONNX Limitations to Be Awere Of
- **Not all scikit-learn models are supported**. Most common ones (linear models, tree-based models, SVMs, pipelines) are. Check the [skl2onnx supported models list](https://onnx.ai/sklearn-onnx/supported.html).
- **Custom Python preprocessing won't convert**. If your Pipeline includes a custom Python transformer (a class you wrote), `skl2onnx` won't know how to convert it.
- **Float32 requirement**. ONNX models typically work with `float32`, while NumPy defaults to `float64`. You need to cast your input data.

### 7.2. Other Portable Formats (Awareness)

ONNX is not the only cross-platform format. Here are others you may encounter:

| Format | Use Case |
| :--- | :--- |
| **ONNX** | General-purpose, broad language support |
| **PMML** | Legacy enterprise systems (XML-based, older standard) |
| **TensorFlow SavedModel** | TensorFlow/Keras models specifically |
| **TorchScript** | PyTorch models specifically |

For a data scientist working primarily with scikit-learn, ONNX is the most relevant portable format.

---

## 8. Summary

| Concept | Key Takeaway |
| :--- | :--- |
| **ONNX** | An open format for representing ML models as framework-agnostic computational graphs. |
| **skl2onnx** | The library that converts scikit-learn models/Pipelines to `.onnx` files. |
| **ONNX Runtime** | A high-performance inference engine that runs `.onnx` files on any platform. |
| **Initial Types** | You must define the input shape and data type when converting (`FloatTensorType`). |
| **float32** | ONNX expects `float32` input — cast with `.astype(np.float32)`. |
| **When to use** | When the production environment is not Python, or when inference speed is critical. |
| **Default choice** | For Python-based projects (most freelance work), **joblib remains the better default**. |

---

**Next section:** [API Development](../02_api_development/) — Wrapping your saved model in a FastAPI web service.